# Hyperviser Construction
## Imports

In [1]:
import pandas as pd

from anthropic          import AsyncAnthropic
from IPython.display    import Markdown, display
from pathlib            import Path

pd.set_option("display.max_colwidth", 80)

In [2]:
from src.trace      import afficher_trace
from src.trace      import test_span
from src.tools      import init

from src.agent      import set_model
from src.agent      import Agent_Web, Agent_Id
from src.agent      import execute

from src.tools      import new_tool
from src.tools      import get_weather
from src.tools      import get_id, get_firstname, get_lastname

from src.tools      import list_id, replace_or_write

## Valeurs Constant

In [3]:
set_model("qwen3:4b", AsyncAnthropic(base_url="http://localhost:11434", api_key="ollama"))

In [4]:
__DATA = Path("data")
__MODEL = __DATA / "model"
__CONTEXT = __DATA / "context"

WEB_AGENT = __MODEL / "web_agent.joblib"
DB_AGENT = __MODEL / "db_agent.joblib"

## DB Creation

In [5]:
init()
list_id()

[{'id': 1,
  'firstname': 'Kylian',
  'lastname': 'Drif',
  'job': 'Concepteur',
  'birthdate': '02/10/2004',
  'email': 'kylian.drif@epita.fr',
  'language': 'fr'}]

## Simulation d'une base de filesystem

## Définition des outils

In [6]:
JOUR = {"type": "string", "description": "date AAAA-MM-JJ"}
HEURE = {"type": "string", "description": "heure HH:MM (24 h)"}

OUTILS = [
    new_tool(get_weather, get_weather.__doc__,
      {
            "latitude": {"type": "string", "description": "La latitude de l'utilisateur."},
            "longitude": {"type": "string", "description": "La longitude de l'utilisateur."},
      }, []),
    new_tool(get_id, get_id.__doc__,
      {
            "id": {"type": "integer", "description": "L'identifiant de l'utilisateur."}
      }, []),
    new_tool(get_firstname, get_firstname.__doc__,
      {
            "firstname": {"type": "string", "description": "Le prénom de l'utilisateur."}
      }, []),
    new_tool(get_lastname, get_lastname.__doc__,
      {
            "lastname": {"type": "string", "description": "Le nom de famille de l'utilisateur."}
      }, []),
    new_tool(replace_or_write, replace_or_write.__doc__,
      {
            "firstname": {"type": "string", "description": "Le prénom de l'utilisateur."},
            "lastname": {"type": "string", "description": "Le nom de famille de l'utilisateur."},
            "job": {"type": "string", "description": "Le travail de l'utilisateur ou la relation que tu as avec lui."},
            "birthdate": {"type": "string", "description": "La date de naissance de l'utilisateur sous la forme DD/MM/YYYY."},
            "email": {"type": "string", "description": "L'email de l'utilisateur pour le contacter."},
            "language": {"type": "string", "description": "La langue de préférence de l'utilisateur sous la forme fr, en, ..."},
      }, []),
]



## Le SPAN d'observation

In [7]:
await test_span()

✓ traceur OK


## L'agent

In [8]:
SYSTEM_V1 = """Tu es un assistant de recherche web. Nous sommes le {aujourd_hui} et tu parles avec l'agent de supervision d'un système multi-agent.

Tu as pour tache de faire 2 choses :
1. Aller chercher des informations concernant la météo.

Si on te demande d'effectuer une action qui ne fait pas partie des taches que tu dois faire, réponds lui qu'il existe un autre agent pour le faire.
Réponds en français, de façon concise."""

assistant_web = Agent_Web("assistant", SYSTEM_V1, version="v1", outils=[OUTILS[0]])

In [9]:
run = await execute(assistant_web, "Quelle est la météo aux coordonnées : lat=45.764975, long=4.789815")
afficher_trace(run.trace_id)
display(Markdown(run.reponse))

invoke_agent assistant                   |████████████████████████████████████| 261.18 s  
  chat qwen3:4b                          |███                                 | 27.36 s  1 → 251 tok · tool_use
    execute_tool get_weather             |   █                                |  0.48 s  {"latitude": "45.764975", "longitude": "4.789815"}
    chat qwen3:4b                        |   ████████████████████████████████ | 233.34 s  354 → 2048 tok · max_tokens


La météo aux coordonnées lat=45.7

In [10]:
run = await execute(assistant_web, "Comment fait on de la soupe au chou ?")
afficher_trace(run.trace_id)
display(Markdown(run.reponse))

invoke_agent assistant                   |████████████████████████████████████| 47.26 s  
  chat qwen3:4b                          |███████████████████████████████████ | 47.26 s  1 → 417 tok · end_turn


Il existe un autre agent pour vous aider à trouver la recette de la soupe au chou.

___

In [11]:
SYSTEM_V1 = """Tu est un assistant de recherche de base de donnée. Nous sommes le {aujourd_hui} et tu parles avec {utilisateur}.

Tu as pour tache de faire 1 choses :
1. Utiliser tes outils pour extraire des informations sur les utilisateurs qui utilisent le réseau d'agent dont tu fait parti.

Si on te demande d'effectuer une action qui ne fait pas partie des taches que tu dois faire, réponds lui qu'il existe un autre agent pour le faire.
Réponds en français, de façon concise."""

assistant_db = Agent_Id("assistant", SYSTEM_V1, version="v1", outils=[OUTILS[1], OUTILS[2], OUTILS[3], OUTILS[4]])

In [12]:
run = await execute(assistant_db, "Qu'est il écrit dans le fichier ./main.cpp")
afficher_trace(run.trace_id)
display(Markdown(run.reponse))

invoke_agent assistant                   |████████████████████████████████████| 36.58 s  
  chat qwen3:4b                          |███████████████████████████████████ | 36.58 s  17 → 333 tok · end_turn


Il n'existe pas d'outil pour accéder au fichier ./main.cpp. Un autre agent peut gérer cette requête.

In [13]:
run = await execute(assistant_db, "Qui est Kylian Drif")
afficher_trace(run.trace_id)
display(Markdown(run.reponse))

invoke_agent assistant                   |████████████████████████████████████| 256.32 s  
  chat qwen3:4b                          |████████████████████                | 145.52 s  13 → 1494 tok · tool_use
    execute_tool get_firstname           |                    █               |  0.00 s  {"firstname": "Kylian"}
    chat qwen3:4b                        |                    ███████████████ | 110.80 s  1568 → 734 tok · end_turn


Kylian Drif est un concepteur né le 02/10/2004 (kylian.drif@epita.fr).

In [14]:
assistant_web.save(WEB_AGENT)
assistant_db.save(DB_AGENT)